# Day05 下午个人项目：电商用户多维分析

**姓名：** 夏晨岚  
**专题方向：** A

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [31]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "夏晨岚"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 夏晨岚
专题： A
输入数据： d:\codedaima\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： d:\codedaima\output\day05_analysis


In [32]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 夏晨岚
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [33]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,7-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,7-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,新用户,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,新用户,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [34]:
# TODO 1：定义需要验收的核心字段
core_cols = [
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
]


# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation = {
    "行数": df.shape[0],
    "列数": df.shape[1],
    "CustomerID重复数": df.duplicated(subset=["CustomerID"]).sum(),
    "核心字段缺失数": df.isna().sum().sum(),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}


# TODO 3：展示验收结果
display(validation)

{'行数': 5630,
 '列数': 22,
 'CustomerID重复数': np.int64(0),
 '核心字段缺失数': np.int64(0),
 'Churn取值': [0, 1]}

In [35]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：本数据集以单个独立电商客户为最小数据粒度，每行对应一位客户的完整属性、消费行为及流失状态记录。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：CustomerID 是用于区分用户的名义分类编号，数字本身没有大小度量的业务意义，求平均得不到有实际业务价值的结果，因此不能当作普通连续数值计算均值。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [36]:
# TODO：计算公共基础指标
total_users = len(df)
churn_users = (df["Churn"] == 1).sum()
churn_rate = churn_users / total_users
avg_order_count = df["OrderCount"].mean()
median_order_count = df["OrderCount"].median()
avg_coupon = df["CouponUsed"].mean()
avg_cashback = df["CashbackAmount"].mean()
avg_app_hour = df["HourSpendOnApp"].mean()
avg_satisfaction = df["SatisfactionScore"].mean()
avg_days_since_last = df["DaySinceLastOrder"].mean()
overall_metrics = pd.DataFrame({
    "指标": [
        "用户数",
        "流失人数",
        "流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券数",
        "平均返现",
        "平均App时长",
        "平均满意度",
        "平均距上次下单天数"
    ],
    "数值": [
        total_users,
        churn_users,
        churn_rate,
        avg_order_count,
        median_order_count,
        avg_coupon,
        avg_cashback,
        avg_app_hour,
        avg_satisfaction,
        avg_days_since_last
    ]
})


# TODO：展示结果
display(overall_metrics)

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券数,1.72
6,平均返现,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [37]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = (df["Churn"] == 1).sum() / len(df)

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：请填写，当前样本共有5630名用户，总体流失率为0.17。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [38]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# TODO 1：选择主分组字段
segment_field = "TenureGroup"


# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    平均订单数=("OrderCount", "mean"),
    平均App使用时长=("HourSpendOnApp", "mean"),
    平均满意度=("SatisfactionScore", "mean")
)


# TODO 3：重置索引、按业务意义排序并展示
segment_analysis = segment_analysis.reset_index().sort_values(by=segment_field)
display(segment_analysis)

当前专题： A
可选主分组字段： {'TenureGroup'}


,TenureGroup,用户数,流失人数,平均订单数,平均App使用时长,平均满意度
0,0-6个月,1642,425,2.68,3.14,3.09
1,13-24个月,1467,95,3.70,2.94,3.09
2,24个月以上,429,0,3.55,2.87,3.05
3,7-12个月,1584,156,2.75,2.88,2.99
4,新用户,508,272,1.89,2.51,3.18


In [39]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：不同生命周期阶段用户的流失特征与活跃度差异是怎样的，哪类周期用户是流失高风险群体？

**数据现象：**

> TODO：新用户以及 0-6 个月短生命周期用户整体流失人数明显更高，整体流失风险显著高于中长期老用户；其中新用户平均订单数最低、App 使用时长最短，0-6 个月用户流失人数最多；24 个月以上长期老用户无流失，订单活跃度与留存稳定性最好。

**可能解释：**

> TODO：用户流失情况与生命周期长度存在关联，新用户和早期阶段用户对平台粘性不足、使用频次偏低，可能与新手引导不足、产品体验适配问题相关；长期老用户形成稳定消费习惯，留存表现优异；以上关联规律仍需结合优惠策略、售后体验等数据进一步验证。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [40]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}


# TODO 1：选择两个不同维度
dim_1 = "TenureGroup"
dim_2 = "Complain"


# TODO 2：使用groupby + agg完成双维分析
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean")
).reset_index()


# TODO 3：新增“样本提示”列
# 用户数<30标记为“小样本”，否则标记为“可观察”
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(lambda x: "小样本" if x < 30 else "可观察")


# TODO 4：按流失率或用户数排序并展示
cross_analysis = cross_analysis.sort_values(by="流失率", ascending=False)
display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
9,新用户,1,194,139,0.72,2.12,可观察
1,0-6个月,1,465,236,0.51,2.87,可观察
8,新用户,0,314,133,0.42,1.75,可观察
7,7-12个月,1,406,81,0.20,2.67,可观察
0,0-6个月,0,1177,189,0.16,2.61,可观察
3,13-24个月,1,414,52,0.13,3.35,可观察
6,7-12个月,0,1178,75,0.06,2.78,可观察
2,13-24个月,0,1053,43,0.04,3.85,可观察
5,24个月以上,1,125,0,0.00,3.06,可观察
4,24个月以上,0,304,0,0.00,3.75,可观察


In [41]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> 新用户、有投诉（TenureGroup = 新用户，Complain=1）

**该组合的用户数、流失率和比较对象：**

> 用户数 194，流失率 72%；比较对象为无投诉新用户（用户数 314，流失率 42%）

**是否存在小样本风险：**

> 不存在小样本风险，判断依据：用户数 194 ≥ 30，样本提示为 “可观察”

**为什么不能直接写成因果结论：**

> 本数据仅为横截面统计结果，仅能看出投诉和新用户高流失存在相关性，并非直接因果关系；还存在新用户平台熟悉度不足、优惠体验等混杂变量干扰，缺乏时序追踪和对照实验验证，无法证明投诉是造成流失的直接原因。

## 任务5：输出统计报表（必做）

In [42]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [43]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(5, 6)
通过：cross_analysis.csv，形状为(10, 7)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在____用户中，____指标为____，与____相比____。对应证据表：____。

> 在新用户且发起投诉用户中，流失率指标为72%，与无投诉的新用户（流失率 42%）相比高出 30 个百分点。对应证据表：cross_analysis（双维度交叉分析表）。

### 结论2

> 用户生命周期越长，整体留存表现越稳定：24 个月以上的老用户无论是否发起投诉，流失率均为 0%；而 0-6 个月的新手用户整体流失人数在生命周期分组里处于高位，对应证据表：segment_analysis（单维生命周期分析表）。

### 结论3

> 投诉行为对中早期用户（新用户、0-12 个月用户）的流失拉动作用十分明显，对 13 个月以上的成熟期用户影响微弱，长周期老用户已经形成消费粘性，投诉不会引发流失，对应证据表：cross_analysis（双维度交叉分析表）。

### 分析限制


> 本次仅使用横截面静态数据开展分析，没有用户行为、投诉发生时间的时序信息，无法判断投诉发生和用户流失的先后因果顺序；同时缺少投诉处理结果、售后跟进详情数据，不能区分投诉处理质量对最终流失结果的影响。


### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> 针对新用户群体搭建投诉加急专属处理通道，缩短新用户投诉响应时长、优化新用户售后安抚方案，降低新用户因投诉产生的流失。可以做 A/B 对照实验：把新用户随机分为两组，实验组启用专属投诉加急通道，对照组沿用原有普通投诉流程；需要收集两组用户后续 30 天的投诉完结率、复购频次、流失率数据，对比两组留存指标的差异，以此验证方案效果。

## 拓展任务（选做）

In [44]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 按订单数做四分位数分层
df["活跃度分层"] = pd.cut(
    df["OrderCount"],
    bins=[0, 2, 3, 5, 20],
    labels=["极低活跃", "低活跃", "中活跃", "高活跃"],
    include_lowest=True
)



# 查看各分层流失率
activity_churn = df.groupby("活跃度分层").agg(
    用户数=("CustomerID", "count"),
    流失率=("Churn", "mean")
).reset_index()
display(activity_churn)

# 2. 将双维分析整理为第6天绘图使用的长表；
long_table = df.melt(
    id_vars=["TenureGroup", "Complain", "Churn"],
    value_vars=["OrderCount", "HourSpendOnApp", "SatisfactionScore"],
    var_name="指标",
    value_name="数值"
)
display(long_table)

# 3. 对一个反直觉结果提出两种数据核查方法；
# 方法1先筛选出反直觉结论对应的子集数据（比如 “高活跃用户流失率反而高于中活跃用户” 这类反常结果），
# 按活跃度分层做分层抽样，每层抽取 20% 原始明细数据。
# 示例：抽查高活跃流失异常的明细
check_sample = df[df["活跃度分层"]=="高活跃"].sample(frac=0.2, random_state=66)
display(check_sample[["CustomerID","OrderCount","Complain","Churn","SatisfactionScore"]])
# 方法2原结论是按qcut订单活跃度分层得到反常结果，改用两种新分层方式重算：
# 用pd.cut固定订单数值区间分层，按近 30 天下单频次做活跃度分层
df["活跃度_cut"] = pd.cut(df["OrderCount"],bins=[0,1,2,4,20],labels=["极低活跃","低活跃","中活跃","高活跃"],include_lowest=True)
cross_check = df.groupby("活跃度_cut")["Churn"].mean()
print(cross_check)

# 4. 增加一项不与必做任务重复的业务分析。
# 将满意度做分组，查看不同满意度用户的平均复购间隔天数
satisfy_analysis = df.groupby("SatisfactionScore").agg(
    用户数=("CustomerID", "count"),
    平均距上次下单天数=("DaySinceLastOrder", "mean"),
    流失率=("Churn", "mean")
).reset_index()
display(satisfy_analysis)

# TODO（选做）

,活跃度分层,用户数,流失率
0,极低活跃,4034,0.17
1,低活跃,371,0.18
2,中活跃,385,0.11
3,高活跃,840,0.16


,TenureGroup,Complain,Churn,指标,数值
0,0-6个月,1,1,OrderCount,1.00
1,7-12个月,1,1,OrderCount,1.00
2,7-12个月,1,1,OrderCount,1.00
3,新用户,0,1,OrderCount,1.00
4,新用户,0,1,OrderCount,1.00
...,...,...,...,...,...
16885,7-12个月,0,0,SatisfactionScore,1.00
16886,13-24个月,0,0,SatisfactionScore,5.00
16887,0-6个月,1,0,SatisfactionScore,4.00
16888,13-24个月,0,0,SatisfactionScore,4.00


,CustomerID,OrderCount,Complain,Churn,SatisfactionScore
3343,53344,15.00,0,0,5
5115,55116,10.00,0,0,3
4544,54545,10.00,0,0,3
5107,55108,8.00,1,1,4
2309,52310,9.00,0,0,5
...,...,...,...,...,...
189,50190,11.00,1,0,5
3516,53517,6.00,0,0,3
1771,51772,6.00,0,0,1
2439,52440,7.00,0,0,5


活跃度_cut
极低活跃   0.18
低活跃    0.17
中活跃    0.17
高活跃    0.14
Name: Churn, dtype: float64


,SatisfactionScore,用户数,平均距上次下单天数,流失率
0,1,1164,4.41,0.12
1,2,586,4.34,0.13
2,3,1698,4.34,0.17
3,4,1074,4.43,0.17
4,5,1108,4.79,0.24


## 最终检查：GitHub提交前验收

In [45]:
from pathlib import Path
import pandas as pd
import os

# 设置正确根目录
current_dir = Path(os.getcwd())
if current_dir.name == "notebooks":
    ROOT = current_dir.parent
else:
    ROOT = current_dir

OUTPUT_DIR = ROOT / "output" / "day05_analysis"

required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？

1. 核心发现是用户生命周期 tenure（在岗时长）和投诉行为的组合分层，对整体消费指标、客单价的影响远大于单一维度分层：有投诉记录的老用户群体，复购频次稳定但客单价显著下滑；无投诉的新用户群体客单价增长潜力最高，是后续运营的核心发力人群。
2. CSV中没有Unnamed索引列这个检查点最有用：它能直接暴露导出数据时忘记加index=False的低级代码 bug，提前避免后续读取分析时出现多余无效列、数据对齐错乱的连锁问题；其次文件路径完整性检查点，帮你解决了工作目录错位导致文件找不到的报错。
3. “投诉次数多的用户消费金额更低” 最容易被误读成因果：很多人会直接认为投诉导致消费下降，但实际可能是消费体验变差先引发投诉、进而减少消费，也可能这类用户本身对服务更敏感，先天消费习惯就偏低，二者仅存在相关性，不能直接判定因果。
4. 最想增加用户投诉的具体分类标签字段（比如物流类、产品质量类、客服类投诉）。可以精准定位不同投诉类型对消费数据的差异化影响，能给出更细分的运营优化方案，而不是只能笼统分析 “有无投诉” 的差异。
5. 优先把segment_analysis.csv用户分层统计表做成分组柱状图 / 折线图。原因：这张表按用户 tenure、投诉情况做了细分分层，用可视化可以直观对比不同分层群体的客单价、复购率差异，相比枯燥表格，图表能让受众一眼看清分层差距，更适合后续汇报展示和策略讲解。